# Data pipeline for Harris WGAN

### Data preparation
First prepare a netCDF-file which contains all relevant data (coarse-input, high-res target and hig-res static data). <br>
Here, we will utilize the validation dataset of the T2m downscaling task.

In [2]:
import os
import xarray as xr

In [31]:
data_dirin = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/with_snow/val"
data_dirout = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/testdata"

fname_in = os.path.join(data_dirin, "downscaling_benchmark_t2m_val.nc")
fname_out = os.path.join(data_dirout, "downscaling_benchmark_t2m_allmem_test.nc")

ds_in = xr.open_dataset(fname_in)

In [32]:
var_in_coa = [var for var in list(ds_in.variables) if var.endswith("_in")]
var_tar_hres = [var for var in list(ds_in.variables) if var.endswith("_tar")]

In [33]:
ds_coa = ds_in[var_in_coa].isel({"rlat": slice(0, -1, 4), "rlon": slice(0, -1, 4)})     # get coarse data from bilinearly interpolated input dataset
ds_hres = ds_in[var_tar_hres]

In [34]:
ds_coa = ds_coa.rename({"rlon": "rlon_in", "rlat": "rlat_in"})
ds_hres = ds_hres.rename({"rlon": "rlon_tar", "rlat": "rlat_tar"})

In [35]:
ds_new = xr.merge([ds_coa, ds_hres])

In [36]:
ds_new.to_netcdf(fname_out)

## Tensorflow data pipeline

In [47]:
import os, glob
from typing import List, Tuple
import gc
import numpy as np
import xarray as xr
import tensorflow as tf

def split_in_tar(ds: xr.Dataset, predictands: List = None, predictors: List = None, static_vars: List = None) -> Tuple[xr.Dataset, xr.Dataset]:
    """
    Split data array with variables-dimension into input and target data for downscaling
    :param da: The unsplitted data array
    :param target_var: Name of target variable which should consttute the first channel
    :param predictands: List of selected predictand variables; parse None to use
                        all predictands (vars with suffix _tar)
    :param predictors: List of selected predictor variables; parse None to use all predictors (vars with suffix _in)
    :return: The split data array.
    """
    varnames = list(ds.data_vars)

    if predictors is None:
        invars = [var for var in varnames if var.endswith("_in")]
    else:
        assert all([predictor in varnames for predictor in
                    predictors]), f"At least one predictor is not a data variable. Available variables are {*varnames,}"
        invars = list(predictors)
    if predictands is None:
        tarvars = [var for var in varnames if var.endswith("_tar")]
    else:
        assert all([predictand in varnames for predictand in
                    predictands]), f"At least one predictor is not a data variable. Available variables are {*varnames,}"
        tarvars = list(predictands)
    
    if static_vars is None:
        ds_in, ds_tar = ds[invars], ds[tarvars]

        return ds_in, ds_tar
    else: 
        assert all([static_var in varnames for static_var in
                    static_vars]), f"At least ostatic high-res is not a data variable. Available variables are {*varnames,}"
        statvars = list(static_vars)
        
        ds_in, ds_tar, ds_stat = ds[invars], ds[tarvars], ds[statvars]

        return ds_in, ds_tar, ds_stat

def reshape_ds(ds):
    """
    Convert a xarray dataset to a data-array where the variables will constitute the last dimension (channel last)
    :param ds: the xarray dataset with dimensions (dims)
    :return da: the data-array with dimensions (dims, variables)
    """
    da = ds.to_array(dim="variables")
    da = da.transpose(..., "variables")
    return da


def make_tf_dataset_allmem(ds: xr.Dataset, batch_size: int, predictands: List, predictors: List, static_vars: List, 
                           lshuffle: bool = True, shuffle_samples: int = 20000, named_targets: bool = False,
                           var_tar2in: str = None, lrepeat: bool = True, drop_remainder: bool = True) -> tf.data.Dataset:
    """
    Build-up TensorFlow dataset from a generator based on the xarray-data array.
    NOTE: All data is loaded into memory
    :param ds: the xarray dataset. Input variable names must carry the suffix '_in', whereas it must be '_tar' for target variables
    :param batch_size: number of samples per mini-batch
    :param predictands: List of selected predictand variables
    :param predictors: List of selected predictor variables; parse None to use all predictors (vars with suffix _in)
    :param lshuffle: flag if shuffling should be applied to dataset
    :param shuffle_samples: number of samples to load before applying shuffling
    :param named_targets: flag if target of TF dataset should be dictionary with named target variables
    :param var_tar2in: name of target variable to be added to input (used e.g. for adding high-resolved topography
                                                                        to the input)
    :param lrepeat: flag if dataset should be repeated
    :param drop_remainder: flag if samples will be dropped in case batch size is not a divisor of # data samples
    :param with_horovod: flag to trigger horovod-based distributed dataset creation
    :param lembed: flag to trigger temporal embedding (not implemented yet!)
    """

    # add time dimension to constant variables
    for var in ds.data_vars:
        if "time" not in ds[var].dims:
            ds[var] = ds[var].expand_dims({"time": ds["time"]}, axis=0)

    ds_in, ds_tar, ds_stat = split_in_tar(ds, predictands=predictands, predictors=predictors, static_vars=static_vars)    

    # convert dataset to data arrays and load into memory
    da_in, da_tar, da_stat = reshape_ds(ds_in).astype("float32", copy=True), \
                             reshape_ds(ds_tar).astype("float32", copy=True), \
                             reshape_ds(ds_stat).astype("float32", copy=True),

    if var_tar2in is not None:
        # NOTE: * The order of the following operation must be the same as in StreamMonthlyNetCDF.getitems
        #       * The following operation order must concatenate var_tar2in by da_in to ensure
        #         that the variable appears at first place. This is required to avoid
        #         that var_tar2in becomes a predeictand when slicing takes place in tf_split
        da_in = xr.concat([da_tar.sel({"variables": var_tar2in}), da_in], "variables")

    varnames_tar = da_tar["variables"].values

    def gen_named(darr_in, darr_tar):
        # darr_in, darr_tar = darr_in.load(), darr_tar.load()
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            tar_now = darr_tar.isel({"time": t})
            yield tuple((darr_in.isel({"time": t}).values,
                            {var: tar_now.sel({"variables": var}).values for var in varnames_tar}))

    def gen_unnamed(darr_in, darr_tar):
        # darr_in, darr_tar = darr_in.load(), darr_tar.load()
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            yield tuple((darr_in.isel({"time": t}).values, darr_tar.isel({"time": t}).values))
        
    def gen_dict(darr_in, darr_tar, darr_stat):
        ntimes = len(darr_in["time"])
        for t in range(ntimes):
            yield tuple(({"lo_res_inputs": darr_in.isel({"time": t}).values, "hi_res_inputs": darr_stat.isel({"time": t}).values},
                         {"output": darr_tar.isel({"time": t}).values}))

    if named_targets is True:
        gen_now = gen_named
    elif static_vars is not None:
        gen_now = gen_dict
    else:
        gen_now = gen_unnamed

    # create output signatures from first sample
    if static_vars is None:
        s0 = next(iter(gen_now(da_in, da_tar)))
        sample_spec_in = tf.TensorSpec(s0[0].shape, dtype=s0[0].dtype)
        if named_targets is True:
            sample_spec_tar = {var: tf.TensorSpec(s0[1][var].shape, dtype=s0[1][var].dtype) for var in varnames_tar}
        else:
            sample_spec_tar = tf.TensorSpec(s0[1].shape, dtype=s0[1].dtype)
    
        # re-instantiate the generator and build TF dataset
        gen_train = gen_now(da_in, da_tar)
        
    else: 
        s0 = next(iter(gen_now(da_in, da_tar, da_stat)))
        
        sample_spec_in = {"lo_res_inputs": tf.TensorSpec(s0[0]["lo_res_inputs"].shape, dtype=s0[0]["lo_res_inputs"].dtype), 
                          "hi_res_inputs": tf.TensorSpec(s0[0]["hi_res_inputs"].shape, dtype=s0[0]["hi_res_inputs"].dtype)}
        
        sample_spec_tar = {"output": tf.TensorSpec(s0[1]["output"].shape, dtype=s0[1]["output"].dtype)}
        
        # re-instantiate the generator and build TF dataset
        gen_train = gen_now(da_in, da_tar, da_stat)
                           
    data_iter = tf.data.Dataset.from_generator(lambda: gen_train, output_signature=(sample_spec_in, sample_spec_tar))

    # Notes:
    # * cache is reuqired to make repeat work properly on datasets based on generators
    #   (see https://stackoverflow.com/questions/60226022/tf-data-generator-keras-repeat-does-not-work-why)
    # * repeat must be applied after shuffle to get varying mini-batches per epoch
    # * batch-size is increased to allow substepping in train_step
    if lshuffle > 1:
        data_iter = data_iter.cache().shuffle(shuffle_samples).batch(batch_size, drop_remainder=drop_remainder)
    else:
        data_iter = data_iter.cache().batch(batch_size, drop_remainder=drop_remainder)

    if lrepeat:
        data_iter = data_iter.repeat()

    # clean-up to free some memory
    # free_mem([da, da_in, da_tar, varnames_tar])
    del ds
    del ds_in
    del ds_tar
    del da_in
    del da_tar
    gc.collect()


    return data_iter

In [48]:
tfds = make_tf_dataset_allmem(ds_new, 32, ["t_2m_tar"], ["t2m_in", "sp_in", "sshf_in", "t115_in"], ["fr_land_tar", "hsurf_tar"]) 

2024-07-05 12:11:04.847086: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2024-07-05 12:11:04.847930: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (jwlogin06.juwels): /proc/driver/nvidia/version does not exist
2024-07-05 12:11:04.857269: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX512F
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [52]:
for i, sample in enumerate(tfds):
    
    inputs, outputs = sample
    
    print(inputs["lo_res_inputs"].shape)
    print(inputs["hi_res_inputs"].shape)
    print(outputs["output"].shape)
    
    if i > 1:
        break
    

(32, 32, 36, 4)
(32, 128, 144, 2)
(32, 128, 144, 1)
(32, 32, 36, 4)
(32, 128, 144, 2)
(32, 128, 144, 1)
(32, 32, 36, 4)
(32, 128, 144, 2)
(32, 128, 144, 1)


2024-07-05 12:14:43.486450: W tensorflow/core/kernels/data/cache_dataset_ops.cc:768] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
